# AI Learning Copilot — pipeline walkthrough

**Repo:** https://github.com/Casildagsf/ironhack-final-project
**Live app:** https://ai-learning-copilot-ironhack-final-project.streamlit.app/

This notebook shows *why* the system is built the way it is. Every number below is
**computed live** when you run the notebook — nothing is pasted in.

It **imports** from `src/` and never redefines anything. The deployed app imports the
same modules, so what you see here is what runs in production.

---

### Why the logic lives in `src/` and not in this notebook

A notebook cannot be deployed, imported, or unit-tested. Streamlit Cloud runs
`streamlit run app/app.py`, which imports `src/`. Any logic written in a cell would have
to be copy-pasted into a `.py` file anyway — creating two copies that drift apart.

Every bug we hit this week was a drift between two definitions of one thing. So the
pipeline is a Python package, and this notebook is the **explanation** of it.

### Prerequisite

The raw transcripts are gitignored (classmates are named in them). Regenerate with:

```bash
bash data/raw/fetch_captions.sh
```


In [1]:
import re
import sys
from pathlib import Path

# src/ modules import each other by bare name, exactly as app/app.py does it.
SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import webvtt

import chunking
from ingestion import CAPTIONS_DIR, JARGON_FIXES, load_all, load_recording
from schemas import EMBED_DIMENSIONS, VIDEO_CHUNK_SIZE, build_citation, format_timestamp, loom_time_param

print("captions:", len(list(CAPTIONS_DIR.glob("*.vtt"))), "transcripts")


captions: 120 transcripts


---
## 1. Ingestion — the corpus was broken before we touched it

Loom's auto-captions are the only transcript we have. Two problems had to be fixed before
any of this could work, and **both were found by reading the data, not by guessing.**

### 1a. Speaker tags

Every cue is wrapped in Loom's `<v 0>…</v>` markup, which would otherwise be embedded as
if it were speech.


In [2]:
vtt_path = sorted(CAPTIONS_DIR.glob("w7d2*.vtt"))[0]

raw_cues = webvtt.read(str(vtt_path))
print("RAW    :", repr(raw_cues[0].raw_text))

recording = load_recording(vtt_path)
print("CLEANED:", repr(recording.cues[0].text))
print()
print("lesson:", recording.lesson_id, "|", recording.title, "|", len(recording.cues), "cues")


RAW    : '<v 0>Uhm, now, day 2 of week 7,</v>'
CLEANED: 'Uhm, now, day 2 of week 7,'

lesson: w7d2 | RAG I - Intro | 703 cues


### 1b. The auto-captions destroyed our central concept

The course's main topic is **RAG**. The transcriber heard **"RAC"**.

This is not cosmetic. A RAG copilot whose corpus barely contains the word "RAG" cannot
answer questions about RAG — the embedding for a question about RAG would have nothing
close to match against.

Counted across all 120 transcripts, before and after cleaning:


In [3]:
RAG = re.compile(r"\bRAG\b")
RAC = re.compile(r"\bRAC\b")

raw_corpus = "".join(
    " ".join(c.text for c in webvtt.read(str(p)))
    for p in sorted(CAPTIONS_DIR.glob("*.vtt"))
)
clean_corpus = " ".join(c.text for r in load_all() for c in r.cues)

for name, text in (("RAW", raw_corpus), ("CLEANED", clean_corpus)):
    print("%-8s RAG=%3d   RAC=%3d" % (name, len(RAG.findall(text)), len(RAC.findall(text))))


RAW      RAG= 16   RAC= 61
CLEANED  RAG= 77   RAC=  0


**That is the whole justification for `JARGON_FIXES`.** Without it, roughly four out of
five mentions of the course's central concept were invisible to retrieval.

The same dictionary repairs the other terms the transcriber mangled:


In [4]:
for pattern, replacement in JARGON_FIXES.items():
    print("%-22s -> %s" % (pattern, replacement))


\bRAC\b                -> RAG
\blang chain\b         -> LangChain
\bline chain\b         -> LangChain
\bkarma\b              -> Chroma
\bhugging face\b       -> HuggingFace
\bnum py\b             -> NumPy
\bpie torch\b          -> PyTorch
\bjupiter\b            -> Jupyter


### 1c. What we deliberately did *not* scrub

Classmates are named throughout the recordings and never agreed to be published, so direct
address is replaced with `[student]`.

**The trap:** a blanket name scrub would have destroyed teaching content, because
**Alice, Bob and Charlie are Python loop examples**, not classmates. So the regex only
matches a *greeting or thanks followed by a name*.

A second subtlety, found the hard way: the greeting must match case-insensitively but the
**name must not** — otherwise `"thanks for"` matches, and real sentences get deleted.


---
## 2. Chunking — the chunk size *is* the timestamp precision

A chunk's timestamp is the start of its **first** cue. So a bigger chunk gives the model
more context, but drops the student further from the moment they asked about.

`VIDEO_CHUNK_SIZE = 1000` characters ≈ 65 seconds of speech. That trade-off is the entire
feature: cite too coarsely and "the exact minute" stops being true.

`chunk_cues()` **never splits inside a cue** — a cue is the smallest unit that carries a
real timestamp, so splitting one would mean inventing a time.


In [5]:
chunks = chunking.chunk_recording(recording)
print(VIDEO_CHUNK_SIZE, "chars/chunk ->", len(chunks), "chunks from", len(recording.cues), "cues")

sample = chunks[3]
print()
print("text[:180]:", sample["text"][:180].replace("\n", " "), "...")
print()
print("start_seconds:", sample["metadata"]["start_seconds"])


1000 chars/chunk -> 46 chunks from 703 cues

text[:180]: [w7d2 · RAG I - Intro] or generating text, or whatever. Or anything like that. Now, this model has been learning patterns from the training data. Any information that is not the tr ...

start_seconds: 214


### The citation, and the detail that would have silently broken everything

`build_citation()` turns chunk metadata into the finished `label` and `url`.

**Loom silently ignores `?t=214` and needs `?t=214s`.** A bare number does not error — the
player just starts at zero. It would have looked like the timestamps were wrong rather
than the format.

This is isolated in one function, which is why the UI never had to know about it:


In [6]:
seconds = sample["metadata"]["start_seconds"]
print("raw seconds     :", seconds)
print("format_timestamp:", format_timestamp(seconds), "   <- what the student reads")
print("loom_time_param :", loom_time_param(seconds), "   <- note the trailing 's'")
print()

citation = build_citation(sample["metadata"])
for key, value in citation.items():
    print("%-14s %s" % (key, value))


raw seconds     : 214
format_timestamp: 3:34    <- what the student reads
loom_time_param : 214s    <- note the trailing 's'

source_type    video
lesson_id      w7d2
label          w7d2 · RAG I - Intro · 3:34
url            https://www.loom.com/embed/4421b00033f74db7afa16c9f874a7487?t=214s
start_seconds  214


`app/app.py` drops that `url` straight into an iframe. **The UI never builds a URL** —
which is why one person could own the pipeline and the other the interface without either
needing to understand the other's half.


---
## 3. Contextual headers — fixing a real retrieval failure

**The bug:** *"How does CLIP work?"* returned the **LangChain** lesson, while
`w8d1 · Multimodal Search Engine with CLIP` never appeared at all.

**The cause:** lesson titles were stored in the metadata but never **embedded**. Metadata
is not searched — only the chunk text is. So the single most identifying words about a
lesson were invisible to retrieval.

**The fix:** prepend `[lesson_id · title]` to the chunk text *before* embedding.


In [7]:
clip_rec = load_recording(sorted(CAPTIONS_DIR.glob("w8d1*.vtt"))[0])
header = chunking.contextual_header(clip_rec)

print("header:", header)
print()
print("embedded text now starts with the lesson title:")
print(chunking.chunk_recording(clip_rec)[5]["text"][:150].replace("\n", " "), "...")


header: [w8d1 · Multi-modal RAG]

embedded text now starts with the lesson title:
[w8d1 · Multi-modal RAG] A summary that you can generate using a modern model, a multimodal model. It's another option is that in the embedding space  ...


The effect is largest exactly where it was needed: a chunk in the middle of a lesson that
never says the lesson's own name now still carries it.

Retrieval today, on the committed index:


In [8]:
from embeddings import FULL_INDEX
from retrieval import get_store, search_with_scores

# Be explicit about which index we are reading. get_store() falls back FULL -> DEV,
# and the dev index holds only 5 lessons, so an accidental fallback would quietly
# change every number below.
store = get_store(FULL_INDEX)
print("index:", FULL_INDEX.name, "|", store._collection.count(), "chunks")
print()

for doc, distance in search_with_scores("How does CLIP work?", k=5, store=store):
    meta = doc.metadata
    print("%.3f  %-6s %s" % (distance, meta["lesson_id"], meta["lesson_title"]))


index: full | 5248 chunks



1.012  w8d1   Multimodal Search Engine with CLIP · Multi-modal RAG
1.076  w8d1   Multimodal Search Engine with CLIP · Multi-modal RAG
1.115  w8d1   Multimodal Search Engine with CLIP · Multi-modal RAG
1.135  w7d3   Recap (LanChain Memory + RAG Pipeline)
1.173  w7d3   Recap (LanChain Memory + RAG Pipeline)


Measured at the time of the fix, the distance to the CLIP lesson improved from **1.175 to
0.809**, and every other test query improved too. That before-number cannot be recomputed
here — it belongs to an index built without headers, which no longer exists.

### A caveat we found while writing this notebook

**The exact distances above vary slightly between runs.** Chroma's HNSW index is an
*approximate* nearest-neighbour structure, so a search can occasionally miss the true
closest chunk.

Running this identical query in four fresh Python processes gave `1.012` three times and
`0.725` once — the same index, the same query vector, a different answer. The `0.725` run
found a chunk the other three missed.

**What this does and does not mean.** The right *lesson* is returned every time, so the
citation a student sees is correct — which is why all 30 evaluation cases pass
consistently. But the specific chunk, and therefore the specific **minute** cited, can
differ between two identical questions. Worth knowing before demoing the same question
twice on stage.

The fix would be raising Chroma's `hnsw:search_ef` at collection-creation time, which
means rebuilding and re-committing the index. Recorded as a known limitation rather than
changed a week before submission.


---
## 4. The relevance cutoff — and why a threshold alone is not enough

`tools.py` sets `RELEVANCE_CUTOFF = 1.3`: retrieved chunks further than that are discarded.
The value was **measured, not guessed**. Here is the measurement, live:


In [9]:
from tools import RELEVANCE_CUTOFF

on_topic = [
    "What is RAG?", "What is an embedding?", "How does CLIP work?",
    "What is overfitting?", "How does backpropagation work?",
    "What is a transformer architecture?", "What is tokenization?", "What is LoRA?",
]
off_topic = [
    "What is the capital of France?",
    "Which Kubernetes operator should I use for autoscaling?",
    "How do I bake sourdough bread?",
    "What did the instructor say about quantum error correction?",
    "Who won the 2022 World Cup?",
    "How do I change a car tyre?",
]

scores = {}
for label, questions in (("ON", on_topic), ("OFF", off_topic)):
    scores[label] = [min(d for _, d in search_with_scores(q, k=5, store=store)) for q in questions]
    for q, d in zip(questions, scores[label]):
        flag = "" if d < RELEVANCE_CUTOFF else "  <- rejected"
        print("%-4s %.3f  %-52s%s" % (label, d, q[:52], flag))

print()
print("cutoff                    : %.2f" % RELEVANCE_CUTOFF)
print("worst on-topic            : %.3f" % max(scores["ON"]))
print("best (closest) off-topic  : %.3f" % min(scores["OFF"]))
print("margin                    : %.3f" % (min(scores["OFF"]) - max(scores["ON"])))


ON   0.832  What is RAG?                                        
ON   0.780  What is an embedding?                               
ON   1.012  How does CLIP work?                                 
ON   0.732  What is overfitting?                                
ON   0.641  How does backpropagation work?                      
ON   0.646  What is a transformer architecture?                 
ON   1.016  What is tokenization?                               
ON   0.907  What is LoRA?                                       


OFF  1.278  What is the capital of France?                      
OFF  1.354  Which Kubernetes operator should I use for autoscali  <- rejected
OFF  1.451  How do I bake sourdough bread?                        <- rejected
OFF  1.038  What did the instructor say about quantum error corr
OFF  1.200  Who won the 2022 World Cup?                         
OFF  1.662  How do I change a car tyre?                           <- rejected

cutoff                    : 1.30
worst on-topic            : 1.016
best (closest) off-topic  : 1.038
margin                    : 0.023


### The honest result: the threshold does **not** separate the two classes

The margin is a few hundredths, and several off-topic questions score *below* the cutoff —
"quantum error correction" lands closest of all, because it is semantically near
**quantization**, which the course really does cover.

**No threshold can fix this.** Moving the cutoff down to reject them would also reject
genuine questions like "How does CLIP work?" that score in the same range.

This is why the design does not rely on the cutoff alone:

1. The **LLM** decides whether the retrieved excerpts actually answer the question, and
   refuses if they do not.
2. **`agent.py` drops every citation whenever the answer is a refusal.** If the model is
   refusing, the retrieved chunks are wrong by definition — so they must not be shown.

Before that second rule existed, refusals shipped with confident-looking citations
pointing at unrelated lessons. That is a worse failure than a wrong answer, because it
*looks* sourced.


---
## 5. End to end

`Copilot.ask()` returns the frozen contract — `{answer, citations}` — and nothing else.
That shape was agreed on day 1, which is what let the UI be built against a mock while the
agent did not yet exist.

Note the citations below: **video and notebook in the same answer**. One question returns
both the minute the instructor explained it and the cell where it was coded.


In [10]:
from agent import Copilot

result = Copilot().ask("Show me the code for splitting documents into chunks with LangChain.")

print(result["answer"])
print()
print("-" * 70)
for c in result["citations"]:
    print("%-9s %s" % (c["source_type"], c["label"]))
    print("          ", c["url"])


The course material provides an example of splitting documents into chunks using LangChain with the `MapReduceDocumentsChain`. Here’s a relevant code snippet:

```python
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model

documents = [
 Document(page_content="Apples are red", metadata={"title": "apple_book"}),
 Document(page_content="Blueberries are blue", metadata={"title": "blueberry_book"}),
 Document(page_content="Bananas are yellow", metadata={"title": "banana_book"}),
]

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# Map
map_template = "Write a concise summary of the following: {docs}."
map_prompt = ChatPromptTemplate([(

---
## What this notebook does not contain

No pipeline logic. Every function called above lives in `src/`, is covered by
`tests/`, and is imported unchanged by the deployed app.

| | |
|---|---|
| `schemas.py` | The frozen contracts — chunk metadata and `{answer, citations}` |
| `ingestion.py` | VTT parsing, jargon repair, name scrubbing |
| `chunking.py` | Cue grouping, timestamps, contextual headers |
| `embeddings.py` | Index building at `dimensions=512` |
| `retrieval.py` | Search over one collection, filtered by `source_type` |
| `notebooks.py` | `.ipynb` parsing into the same chunk shape |
| `tools.py` | The five agent tools |
| `agent.py` | `AgentExecutor`, memory, refusal handling |

## Known limitations

1. **Notebook coverage is weeks 7–8 only** — 158 of 5,248 chunks. Questions about
   LangChain, RAG, memory and CLIP return code citations; questions about embeddings,
   pandas or transformers return video only.
2. **Retrieval is approximate** (§3) — the cited lesson is stable, the cited minute can
   vary between two identical questions.
3. **The relevance cutoff does not separate on- from off-topic questions** (§4). Refusal
   depends on the model judging the retrieved context, backed by dropping citations
   whenever the answer is a refusal.
4. **Two recordings are deliberately excluded** — `w3d4-d` is standup admin and `w6d2-a`
   is the Project-3 homework briefing. Neither is teaching material, so the corpus is
   120 of 120 *teaching* recordings.
